<a href="https://colab.research.google.com/github/swatian1989/coda-public-3d/blob/main/notebooks/RUN_EVERYTHING_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CODA on public data: the whole pipeline in Colab

Runs both arms end to end.

| Arm | Data | Stages |
|---|---|---|
| 1 | mouse prostate, consecutive sections | 1, 2, 5, 6, 7 (the 3D stages) |
| 2 | TCGA-BRCA diagnostic slides | 3, 7, stereology |
| 4 | BCSS annotations | semantic segmentation, needs the GPU |

## Before you start

**Runtime, Change runtime type, T4 GPU, Save.** Only stage 4 needs it, but
setting it now avoids restarting later.

## What Colab imposes, and how this notebook handles it

A free session ends after about 12 hours and disconnects when idle, and the
disk is wiped when it does. The full prostate archive is 63.79 GB and its
download service ignores range requests, so it cannot resume: a disconnect
half way through means starting again.

Two things follow, and both are set up in cell 2. Results are written to your
Google Drive rather than the session disk, so a disconnect costs time and not
work. And `N_SECTIONS` defaults to a subset of the series rather than all 260,
which is what makes the arm finish inside one session. Set it to `None` for the
full series only if you are willing to restart on a disconnect.

**Run cells in order.** Each prints what it found, and stops with an
explanation rather than continuing on empty data.

In [1]:
#@title 1. Environment and code { display-mode: "form" }
import subprocess, sys, os, shutil
from pathlib import Path

REPO = 'https://github.com/swatian1989/coda-public-3d.git'
WORK = Path('/content/coda')
if not WORK.exists():
    subprocess.run(['git', 'clone', '-q', REPO, str(WORK)], check=True)
os.chdir(WORK)
sys.path.insert(0, str(WORK / 'src'))

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'tifffile', 'zarr', 'scikit-image', 'gdown',
                'segmentation-models-pytorch'], check=False)

import torch
gpu = torch.cuda.is_available()
print('working dir :', os.getcwd())
print('GPU         :', torch.cuda.get_device_name(0) if gpu else 'NONE')
if not gpu:
    print('  Arms 1 and 2 run fine on CPU. Stage 4 needs a GPU:')
    print('  Runtime, Change runtime type, T4 GPU, then rerun this cell.')
free = shutil.disk_usage('/content').free / 1e9
print(f'free disk   : {free:.0f} GB')

working dir : /content/coda
GPU         : Tesla T4
free disk   : 70 GB


In [2]:
#@title 2. Settings and Google Drive { display-mode: "form" }
#@markdown Results are written to Drive so a disconnect does not lose them.
USE_DRIVE = True  #@param {type:"boolean"}
#@markdown Sections of the prostate series to analyse. 260 is the full block
#@markdown (1.3 mm) but needs the whole 63.79 GB archive streamed, which cannot
#@markdown resume if the session drops. 60 finishes comfortably in one session.
N_SECTIONS = 60  #@param {type:"integer"}
#@markdown TCGA slides. Each is about 1 GB.
N_TCGA = 6  #@param {type:"integer"}

from pathlib import Path
import os

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE = Path('/content/drive/MyDrive/coda_results')
else:
    SAVE = Path('/content/coda_results')
SAVE.mkdir(parents=True, exist_ok=True)

# keep results on Drive, keep bulky raw data on the session disk
for d in ('results', 'figures', 'reports'):
    src = Path('/content/coda') / d
    dst = SAVE / d
    dst.mkdir(parents=True, exist_ok=True)
    if src.is_symlink():
        src.unlink()
    elif src.exists():
        for f in src.rglob('*'):
            if f.is_file():
                (dst / f.relative_to(src)).parent.mkdir(parents=True, exist_ok=True)
                f.replace(dst / f.relative_to(src))
        import shutil as _s; _s.rmtree(src, ignore_errors=True)
    src.symlink_to(dst)

print('results are saved to', SAVE)
print(f'prostate sections: {N_SECTIONS} | TCGA slides: {N_TCGA}')

Mounted at /content/drive
results are saved to /content/drive/MyDrive/coda_results
prostate sections: 60 | TCGA slides: 6


---
# Arm 2 first: TCGA-BRCA, human breast

Run before the prostate because it is quick, and because finishing one arm
before starting a long unresumable download means a disconnect costs less.

Stages 3 and 7 only. TCGA has no consecutive sections, verified against the GDC
API where the maximum for any patient is seven slides from different blocks, so
the three-dimensional stages cannot run here and are not attempted.

In [ ]:
#@title 3. Download TCGA slides
!python scripts/fetch_tcga_brca.py --n {N_TCGA} --max-gb 12

2026-08-16 15:02:01,109 restricted to BCSS-annotated cases: 154 of 1133 slides remain
2026-08-16 15:02:01,109 GDC reports 1133 diagnostic slides, 1075 MB median; sampling 6 at random (seed 0), not by size
2026-08-16 15:02:01,113 6 patients, 6.2 GB total, 1316 MB median
2026-08-16 15:02:01,113 [1/6] TCGA-AC-A6IW-01Z-00-DX1.C4514189-E64F-4603-8 (123 MB)
2026-08-16 15:03:04,930 [2/6] TCGA-GM-A2DF-01Z-00-DX1.CD0BE6D7-2DB3-4193-8 (1351 MB)
2026-08-16 15:13:38,202 [3/6] TCGA-A2-A0CM-01Z-00-DX1.AC4901DE-4B6D-4185-B (932 MB)
2026-08-16 15:21:02,100 [4/6] TCGA-A7-A6VV-01Z-00-DX1.07AE0E16-A883-4C86-B (354 MB)
2026-08-16 15:23:48,580 [5/6] TCGA-A2-A0SX-01Z-00-DX1.219A994C-8974-4458-9 (1316 MB)
2026-08-16 15:33:52,932 [6/6] TCGA-EW-A1P8-01Z-00-DX1.E9852193-8CDD-49EF-B (2167 MB)


In [ ]:
#@title 4. Arm 2 analysis: cell detection and fibre alignment
!python scripts/run_tcga_analysis.py --max-slides {N_TCGA} --tiles-per-slide 12

---
# Arm 1: serial prostate, the three-dimensional stages

The next cell streams the 63.79 GB archive. The prostate series sits after the
liver inside it, so roughly the first 13 GB yields nothing; that is expected and
not a fault. It keeps only prostate members, so peak disk is the series rather
than the archive.

**This is the long step.** With `N_SECTIONS` set to 60 you can stop the download
once that many sections have arrived, using the cell after it.

In [ ]:
#@title 5. Stream the prostate series (long; stops itself at N_SECTIONS)
import subprocess, threading, time
from pathlib import Path

SEC = Path('/content/coda/data/raw/kartasalo_prostate/extracted/Data_to_IDA/prostate')
proc = subprocess.Popen([sys.executable, '-u', 'scripts/fetch_kartasalo_prostate.py'],
                        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print('streaming; the first ~13 GB is the liver portion and keeps nothing')
try:
    while proc.poll() is None:
        n = len(list(SEC.glob('*.tif'))) if SEC.exists() else 0
        print(f'\r  sections: {n}/{N_SECTIONS}', end='')
        if N_SECTIONS and n >= N_SECTIONS:
            proc.terminate()
            print(f'\n  reached {N_SECTIONS} sections, stopping the download')
            break
        time.sleep(20)
except KeyboardInterrupt:
    proc.terminate(); print('\n  stopped by user')
n = len(list(SEC.glob('*.tif'))) if SEC.exists() else 0
print(f'sections on disk: {n}')
assert n >= 10, ('Too few sections to reconstruct anything. If this stayed at 0, '
                 'the download did not reach the prostate portion of the archive.')

In [ ]:
#@title 6. Arm 1: registration, volume, connectivity, z-resolution
!python scripts/run_prostate_pipeline.py --downsample 16 --limit {N_SECTIONS}

---
# Stage 4: semantic segmentation

The only stage that needs the GPU. Training data is BCSS, over 20,000 tissue
annotations on TCGA-BRCA slides, so the model is validated on the same cohort it
is applied to.

**Pixel 0 is outside the annotated region and 7 is exclude. Neither is an
"other tissue" class.** Training on them teaches the model that unannotated
background is a tissue type, which then appears confidently across whole-slide
inference. Both are given zero weight, and every metric is computed only over
annotated pixels.

In [ ]:
#@title 7. Train the segmentation model (GPU)
#@markdown Opens the standalone notebook's steps inline. Skip if you have
#@markdown already trained and saved weights to Drive.
SKIP_IF_TRAINED = True  #@param {type:"boolean"}
from pathlib import Path
import shutil

MODEL = Path('/content/coda/data/models/stage4_deeplab.pt')
MODEL.parent.mkdir(parents=True, exist_ok=True)
saved = SAVE / 'stage4_deeplab.pt'
if SKIP_IF_TRAINED and saved.exists():
    shutil.copy(saved, MODEL)
    print('using the model already saved on Drive:', saved)
else:
    print('Open notebooks/stage4_segmentation_colab.ipynb and run it, then')
    print('copy stage4_deeplab.pt to', saved)
    print('Re-run this cell afterwards. Training is separated because it takes')
    print('hours and should not be repeated every time the pipeline is run.')

In [ ]:
#@title 8. Apply the segmentation to the TCGA slides (CPU)
# subprocess rather than a ! magic, because a shell escape inside a conditional
# is awkward to read and behaves differently between IPython versions.
import subprocess, sys
from pathlib import Path

if Path('/content/coda/data/models/stage4_deeplab.pt').exists():
    subprocess.run([sys.executable, 'scripts/run_tcga_segmentation.py',
                    '--max-slides', str(N_TCGA)], cwd='/content/coda')
else:
    print('No trained model, so tissue composition is not computed.')
    print('Everything else in both arms is unaffected: stages 1, 2, 5, 6 and 7')
    print('do not depend on it. Run cell 7 to train, then rerun this cell.')

In [ ]:
#@title 9. Build the report
!python scripts/run_report.py
from pathlib import Path
rep = Path('/content/coda/reports/analysis_report.html')
print('report:', rep, f'{rep.stat().st_size/1e6:.1f} MB' if rep.exists() else 'MISSING')
print('saved to Drive at', SAVE / 'reports')

In [ ]:
#@title 10. Show the results
import json, pandas as pd
from pathlib import Path
from IPython.display import display, HTML, Image

for name, path in (('Arm 1, prostate', 'results/prostate/summary.json'),
                   ('Arm 2, TCGA', 'results/tcga/summary.json'),
                   ('Stage 4', 'results/tcga/stage4_summary.json')):
    p = Path('/content/coda') / path
    if p.exists():
        print(f'--- {name} ---')
        print(json.dumps(json.loads(p.read_text()), indent=2)[:1200], '\n')

for f in sorted(Path('/content/coda/figures').glob('*.png')):
    display(Image(filename=str(f)))

rep = Path('/content/coda/reports/analysis_report.html')
if rep.exists():
    from google.colab import files
    files.download(str(rep))